<a href="https://colab.research.google.com/github/Alchwalch/Deep-Learning-Study/blob/main/Image%20Detection/AlexNet_pytorch.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [20]:
import matplotlib.pyplot as plt
import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
import torchvision
import torchvision.transforms as transforms
import torchvision.datasets as datasets
from torch.utils.data import Subset,DataLoader,random_split
from tqdm.notebook import tqdm
from torch.optim.lr_scheduler import ReduceLROnPlateau

In [21]:
from google.colab import drive
import os
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [22]:
epochs=30
start_epoch=0
learning_rate=1e-3
batch_size=32
best_loss=float('inf')
patience = 5
no_improve = 0
device='cuda' if torch.cuda.is_available() else 'cpu'

In [23]:
class PCAColorAugmentation:
    def __init__(self, eigenvectors, eigenvalues, std=0.1):
        self.eigenvectors = eigenvectors
        self.eigenvalues = eigenvalues
        self.std = std

    def __call__(self, img):
        img = img.permute(1, 2, 0)
        alphas = torch.normal(0, self.std, size=(3,))
        delta = self.eigenvectors @ (alphas * self.eigenvalues)
        img = (img + delta).clamp(0, 1)
        return img.permute(2, 0, 1)

In [24]:
def compute_pca(dataset, n_samples=10000):
    loader = DataLoader(dataset, batch_size=n_samples, shuffle=True)
    imgs, _ = next(iter(loader))
    pixels = imgs.permute(0,2,3,1).reshape(-1, 3)
    cov = torch.cov(pixels.T)
    eigenvalues, eigenvectors = torch.linalg.eigh(cov)
    return eigenvectors, eigenvalues

In [25]:
raw_train_dataset = datasets.STL10(root='/content/drive/MyDrive/data', split='train', download=True, transform=transforms.ToTensor())
eigenvectors, eigenvalues = compute_pca(raw_train_dataset)

transform_train = transforms.Compose([
    transforms.Resize((256, 256)),
    transforms.RandomCrop(224),
    transforms.RandomHorizontalFlip(),
    transforms.ToTensor(),
    PCAColorAugmentation(eigenvectors, eigenvalues, std=0.1),
])

transform_test = transforms.Compose([
    transforms.Resize((256, 256)),
    transforms.CenterCrop(224),
    transforms.ToTensor(),
])

train_dataset = datasets.STL10(root='/content/drive/MyDrive/data', split='train', download=True, transform=transform_train)
test_dataset = datasets.STL10(root='/content/drive/MyDrive/data', split='test', download=True, transform=transform_test)

#블로그 리뷰용 시드 고정
generator = torch.Generator().manual_seed(42)

train_dataset, val_dataset = random_split(train_dataset, [4000, 1000], generator=generator)
test_subset = Subset(test_dataset, range(1000))


train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=batch_size, shuffle=False)
test_loader = DataLoader(test_subset, batch_size=batch_size, shuffle=False)

In [26]:
class Seq2(nn.Module):
  def __init__(self,device='cpu'):
    super(Seq2,self).__init__()
    self.device=device
    self.seq2 = nn.Sequential(
        nn.Conv2d(48, 128, kernel_size=5, padding=2),
        nn.ReLU(),
        nn.LocalResponseNorm(5, alpha=1e-4, beta=0.75, k=2.0),
        nn.MaxPool2d(kernel_size=3, stride=2)
    )

  def forward(self,x):
    return self.seq2(x)

In [27]:
class Seq4(nn.Module):
  def __init__(self,device='cpu'):
    super(Seq4,self).__init__()
    self.device=device
    self.seq4 = nn.Sequential(
        nn.Conv2d(192, 192, kernel_size=3, padding=1),
        nn.ReLU(),
        nn.Conv2d(192, 128, kernel_size=3, padding=1),
        nn.ReLU(),
        nn.MaxPool2d(kernel_size=3, stride=2)
    )

  def forward(self,x):
    return self.seq4(x)

In [28]:
class AlexNet(nn.Module):
  def __init__(self, output_size=1000,device='cpu'):
    super(AlexNet,self).__init__()
    self.device=device

    # seq1
    self.seq1 = nn.Sequential(
        nn.Conv2d(3, 96, kernel_size=11, stride=4, padding=0),
        nn.ReLU(),
        nn.LocalResponseNorm(5, alpha=1e-4, beta=0.75, k=2.0),
        nn.MaxPool2d(kernel_size=3, stride=2)
    )

    # seq2_1, seq2_2
    self.seq2_1 = Seq2(device=device)
    self.seq2_2 = Seq2(device=device)

    # seq3
    self.seq3 = nn.Sequential(
        nn.Conv2d(256, 384, kernel_size=3, padding=1),
        nn.ReLU()
    )

    # seq4_1, seq4_2
    self.seq4_1 = Seq4(device=device)
    self.seq4_2 = Seq4(device=device)

    # seq5
    self.seq5 = nn.Sequential(
        nn.Dropout(0.5),
        nn.Linear(6400, 4096),
        nn.ReLU(),
        nn.Dropout(0.5),
        nn.Linear(4096, 4096),
        nn.ReLU(),
        nn.Linear(4096, output_size)
    )

  def forward(self, x):
    x = self.seq1(x)

    x1 = x[:, :48, :, :]
    x2 = x[:, 48:, :, :]
    x1 = self.seq2_1(x1)
    x2 = self.seq2_2(x2)
    x = torch.cat([x1, x2], dim=1)

    x = self.seq3(x)

    x1 = x[:, :192, :, :]
    x2 = x[:, 192:, :, :]
    x1 = self.seq4_1(x1)
    x2 = self.seq4_2(x2)
    x = torch.cat([x1, x2], dim=1)

    x = x.view(x.size(0), -1)
    x = self.seq5(x)
    return x

In [29]:
model = AlexNet(output_size=10,device=device)
model.to(device)
optimizer = torch.optim.Adam(model.parameters(), lr=1e-4)
criterion = nn.CrossEntropyLoss()
scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode='min', patience=3, factor=0.1)

In [35]:
train_loss_list = []
val_loss_list = []
test_acc_list=[]

In [34]:
%%script echo skip

if os.path.exists('/content/drive/MyDrive/alexnet_best.pth'):
  checkpoint = torch.load('/content/drive/MyDrive/alexnet_best.pth')
  model.load_state_dict(checkpoint['model_state_dict'])
  optimizer.load_state_dict(checkpoint['optimizer_state_dict'])
  scheduler.load_state_dict(checkpoint['scheduler_state_dict'])
  train_loss_list = checkpoint['train_loss_list']
  val_loss_list = checkpoint['val_loss_list']
  test_acc_list = checkpoint['test_acc_list']
  best_loss = checkpoint['best_loss']
  start_epoch = checkpoint['epoch']+1

In [33]:
for epoch in tqdm(range(start_epoch,epochs),desc="Epoch"):

  train_loss=0.0
  val_loss=0.0
  test_acc=0.0

  model.train()
  for x,t in tqdm(train_loader,desc="Iteration",leave=False):
    x, t = x.to(device), t.to(device)
    optimizer.zero_grad()
    loss = criterion(model(x), t)
    train_loss+=loss.item()
    loss.backward()
    optimizer.step()

  model.eval()
  with torch.no_grad():
      for x, t in val_loader:
          x, t = x.to(device), t.to(device)
          output = model(x)
          val_loss += criterion(output, t).item()

      for x, t in test_loader:
          x, t = x.to(device), t.to(device)
          output = model(x)
          test_acc += (output.argmax(1) == t).sum().item()

  train_loss /= len(train_loader)
  val_loss /= len(val_loader)
  test_acc /= len(test_subset)

  train_loss_list.append(train_loss)
  val_loss_list.append(val_loss)
  test_acc_list.append(test_acc)

  scheduler.step(val_loss)

  if val_loss < best_loss:
    best_loss=val_loss
    no_improve=0
    torch.save({
      'epoch': epoch,
      'model_state_dict': model.state_dict(),
      'optimizer_state_dict': optimizer.state_dict(),
      'scheduler_state_dict': scheduler.state_dict(),
      'best_loss': best_loss,
      'train_loss_list': train_loss_list,
      'val_loss_list': val_loss_list,
      'test_acc_list': test_acc_list,
    },'/content/drive/MyDrive/alexnet_best.pth')

  else:
    no_improve += 1
    if no_improve >= patience:
      print(f"Early Stopping at Epoch {epoch+1}")
      break

  print(f'Epoch {epoch+1} | Test Acc: {test_acc:.4f}')


Epoch:   0%|          | 0/30 [00:00<?, ?it/s]

Iteration:   0%|          | 0/125 [00:00<?, ?it/s]

KeyboardInterrupt: 

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

axes[0].plot(train_loss_list, label='Train Loss')
axes[0].plot(val_loss_list, label='Validation Loss')
axes[0].set_title("Loss")
axes[0].set_xlabel("Epoch")
axes[0].set_ylabel("Loss")
axes[0].legend()

axes[1].plot(test_acc_list)
axes[1].set_title("Test Accuracy")
axes[1].set_xlabel("Epoch")
axes[1].set_ylabel("Accuracy")
axes[1].legend()

plt.tight_layout()
plt.show()